# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [ ]:
# Initialization

load_dotenv(override=True)

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
if openrouter_api_key:
    print(f"Openrouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("Openrouter API Key not set")
if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set")
if gemini_api_key:
    print(f"Gemini API Key exists and begins {gemini_api_key[:8]}")
else:
    print("Gemini API Key not set")
    
groq_url = "https://api.groq.com/openai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key) 
gemini = OpenAI(base_url=gemini_url, api_key=gemini_api_key)
DB = "prices.db"

Openrouter API Key exists and begins sk-or-v1
Groq API Key exists and begins gsk_ExjO
Gemini API Key not set


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [5]:
get_ticket_price("Paris")

DATABASE TOOL CALLED: Getting price for Paris


'Ticket price to Paris is $899.0'

In [6]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [7]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model='openai/gpt-oss-120b', messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [8]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model='openai/gpt-oss-120b', messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model='openai/gpt-oss-120b', messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [9]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [10]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## A bit more about what Gradio actually does:

1. Gradio constructs a frontend Svelte app based on our Python description of the UI
2. Gradio starts a server built upon the Starlette web framework listening on a free port that serves this Svelte app
3. Gradio creates backend routes for our callbacks, like chat(), which calls our functions

And of course when Gradio generates the frontend app, it ensures that the the Submit button calls the right backend route.

That's it!

It's simple, and it has a result that feels magical.

# Let's go multi-modal!!

We can use gpt-image-1-mini, the image generation model behind GPT-5, to make us some images

Let's put this in a function called artist.

### Price alert: each time I generate an image it costs about 3 cents - don't go crazy with images!

In [11]:
# Some imports for handling images

import base64
from io import BytesIO

from PIL import Image

In [22]:
import os
import base64
from io import BytesIO
from google import genai
from google.genai.types import GenerateContentConfig, Modality
from PIL import Image

# 1. Initialize the official client
client = genai.Client(api_key=gemini_api_key)

def artist(city):
    # 2. Point to the alternative legacy image model variant
    response = client.models.generate_content(
        model="gemini-2.5-flash-image", 
        contents=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
        config=GenerateContentConfig(
            response_modalities=[Modality.IMAGE]
        )
    )
    
    # 3. Process the response parts
    for part in response.candidates.content.parts:
        if part.inline_data:
            return Image.open(BytesIO(part.inline_data.data))
            
    raise ValueError("No image data returned.")

# 4. Execute
image = artist("New York City")
display(image)


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-flash-preview-image, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.5-flash-preview-image, description: Generate content requests per minute limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, m

In [ ]:
image = artist("New York City")
display(image)

BadRequestError: Error code: 400 - {'error': {'code': 400, 'message': 'Please pass a valid API key', 'status': 'INVALID_ARGUMENT'}}

In [18]:
def talker(message):
    response = openrouter.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    return response.content

## Let's bring this home:

1. A multi-modal AI assistant with image and audio generation
2. Tool callling with database lookup
3. A step towards an Agentic workflow


In [23]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openrouter.chat.completions.create(model='nvidia/nemotron-3.5-lightning:free', messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openrouter.chat.completions.create(model='nvidia/nemotron-3.5-lightning:free', messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image


In [24]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

## The 3 types of Gradio UI

`gr.Interface` is for standard, simple UIs

`gr.ChatInterface` is for standard ChatBot UIs

`gr.Blocks` is for custom UIs where you control the components and the callbacks

In [28]:
import base64
import os
import tempfile
import urllib.parse
from io import BytesIO
import gradio as gr
from gtts import gTTS
from openai import OpenAI
from PIL import Image
import requests
import re

# 1. Client setup using OpenRouter for free chat completions
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
MODEL = "openrouter/free"

system_message = (
    "You are a helpful travel assistant. If a user asks about tickets or travel"
    " to a city, use the get_ticket_price tool."
)


# Mock tool definition for the context
def get_ticket_price(city):
  return f"The flight ticket to {city} is $350."


tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_price",
            "description": "Get the ticket price for a destination city",
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_city": {
                        "type": "string",
                        "description": "The city name",
                    }
                },
                "required": ["destination_city"],
            },
        },
    }
]


# 2. Free Image Generation using Pollinations.ai API with Error Handling
def artist(city):
  if not city or not isinstance(city, str):
      return None
      
  # Clean up city names if the LLM hallucinated markdown punctuation
  city = re.sub(r'[^a-zA-Z\s]', '', city).strip()
  
  prompt = (
      f"A highly detailed travel illustration of {city} featuring its most iconic"
      " landmarks, rendered in a bold pop-art style combined with intricate"
      " landmarks, rendered in a striking post-impressionist oil painting style"
      " mixed with woodcut engraving textures. The image must feature swirling,"
      " textured brushstrokes for the clouds and water, bold, rich ink outlines"
      " for all architecture and objects, and a vibrant, contrasting color palette."
      " Include intricate decorative line-work patterns in the sky and buildings."
      " Ensure all key landmarks are prominently featured in this intricate,"
      " hand-crafted artistic style."
  )
  encoded_prompt = urllib.parse.quote(prompt)
  url = f"https://image.pollinations.ai/prompt/{encoded_prompt}?width=1024&height=1024&nologo=true&seed={os.urandom(4).hex()}"
  
  try:
      response = requests.get(url, timeout=15)
      if response.status_code != 200:
          print(f"Pollinations Error ({response.status_code}): {response.text}")
          return None
      
      # Verify that the response content type is actually an image
      if "image" not in response.headers.get("Content-Type", ""):
          print("Pollinations API did not return a valid image type.")
          return None
          
      return Image.open(BytesIO(response.content))
  except Exception as e:
      print(f"Failed to fetch or open image: {e}")
      return None


# 3. Free Text-to-Speech using gTTS
def talker(message):
  # Clean up markdown symbols, pipes, and excessive dashes so TTS doesn't read them out loud
  clean_message = re.sub(r"[-|#*_`~]", " ", message)
  clean_message = re.sub(r"\s+", " ", clean_message).strip()  # Remove extra spaces
  tts = gTTS(text=clean_message, lang="en")
  fp = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
  tts.save(fp.name)
  return fp.name


def chat(history):
  history = [{"role": h["role"], "content": h["content"]} for h in history]
  messages = [{"role": "system", "content": system_message}] + history
  response = client.chat.completions.create(
      model=MODEL, messages=messages, tools=tools
  )
  latest_city = None

  while response.choices[0].finish_reason == "tool_calls":
    message = response.choices[0].message
    responses, tool_city = handle_tool_calls_and_return_latest_city(message)
    if tool_city:
        latest_city = tool_city
    messages.append(message)
    messages.extend(responses)
    response = client.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )

  reply = response.choices[0].message.content
  history += [{"role": "assistant", "content": reply}]

  voice = talker(reply)

  # Only generate a fresh image if a new city was called in this turn
  image = artist(latest_city) if latest_city else None

  return history, voice, image


def handle_tool_calls_and_return_latest_city(message):
  responses = []
  latest_city = None
  for tool_call in message.tool_calls:
    if tool_call.function.name == "get_ticket_price":
      import json
      try:
          arguments = json.loads(tool_call.function.arguments)
          latest_city = arguments.get("destination_city")
          price_details = get_ticket_price(latest_city)
          responses.append({
              "role": "tool",
              "content": price_details,
              "tool_call_id": tool_call.id,
          })
      except Exception as e:
          print(f"Error parsing tool arguments: {e}")
  return responses, latest_city


def put_message_in_chatbot(message, history):
  return "", history + [{"role": "user", "content": message}]


# UI definition
with gr.Blocks() as ui:
  with gr.Row():
    chatbot = gr.Chatbot(height=500, type="messages")
    image_output = gr.Image(height=500, interactive=False)
  with gr.Row():
    audio_output = gr.Audio(autoplay=True)
  with gr.Row():
    message = gr.Textbox(label="Chat with our AI Assistant:")

  message.submit(
      put_message_in_chatbot,
      inputs=[message, chatbot],
      outputs=[message, chatbot],
  ).then(chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output])

ui.launch(inbrowser=True, auth=("ed", "bananas"))


* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Pollinations Error (402): {"status":402,"success":false,"error":{"code":"FREE_LIMIT","message":"Free limit reached: 1 image every 45 seconds per IP. Retry after 1s, or get unlimited images with a pollen key at https://enter.pollinations.ai/?ref=classic","timestamp":"2026-11-01T05:58:07.669Z","fixUrl":"https://enter.pollinations.ai/?ref=classic"}}
Pollinations Error (503): {"status":503,"success":false,"error":{"code":"SERVICE_UNAVAILABLE","message":"The Classic API is at its limit right now. Retry after 10s, or get full access to every model with a pollen key at https://enter.pollinations.ai/?ref=classic","timestamp":"2026-11-01T05:59:49.361Z","fixUrl":"https://enter.pollinations.ai/?ref=classic"}}
Pollinations Error (503): {"status":503,"success":false,"error":{"code":"SERVICE_UNAVAILABLE","message":"The Classic API is at its limit right now. Retry after 10s, or get full access to every model with a pollen key at https://enter.pollinations.ai/?ref=classic","timestamp":"2026-11-01T06:0

# Exercises and Business Applications

Add in more tools - perhaps to simulate actually booking a flight. A student has done this and provided their example in the community contributions folder.

Next: take this and apply it to your business. Make a multi-modal AI assistant with tools that could carry out an activity for your work. A customer support assistant? New employee onboarding assistant? So many possibilities! Also, see the week2 end of week Exercise in the separate Notebook.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a HUGE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>